In [ ]:
# (a) Train Logistic Regression and XGBoost

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score

from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv("dataset_prt.csv")

df.head()

,Age,Education_Years,Experience_Years,Monthly_Income,Annual_Income,Credit_Score,Spending_Index,Region,Transaction_Date,Loan_Risk
0,22,13,3,60518,726356,617,61.34,SemiUrban,2020-01-18 00:00:00,1
1,48,19,23,60276,717901,572,64.02,Rural,2022-09-18 00:00:00,?
2,54,17,31,25015,301111,604,25.91,?,2020-10-26 00:00:00,0
3,58,10,?,38503,460130,580,37.16,Rural,2019-07-13 00:00:00,0
4,37,?,16,60278,719539,685,56.7,SemiUrban,2021-05-15 00:00:00,?


In [ ]:
df.info()

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               320 non-null    float64
 1   Education_Years   320 non-null    float64
 2   Experience_Years  320 non-null    float64
 3   Monthly_Income    320 non-null    float64
 4   Annual_Income     320 non-null    float64
 5   Credit_Score      320 non-null    float64
 6   Spending_Index    320 non-null    float64
 7   Region            292 non-null    object 
 8   Transaction_Date  288 non-null    object 
 9   Loan_Risk         320 non-null    float64
dtypes: float64(8), object(2)
memory usage: 25.1+ KB


,0
Age,0
Education_Years,0
Experience_Years,0
Monthly_Income,0
Annual_Income,0
Credit_Score,0
Spending_Index,0
Region,28
Transaction_Date,32
Loan_Risk,0


In [ ]:
X = df.drop(['Loan_Risk','Region','Transaction_Date'], axis=1)
y = df['Loan_Risk']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
#Train Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:,1]

auc_log = roc_auc_score(y_test, y_prob_log)
f1_log = f1_score(y_test, y_pred_log)

print("Logistic Regression AUC:", auc_log)
print("Logistic Regression F1 Score:", f1_log)

Logistic Regression AUC: 0.9682051282051283
Logistic Regression F1 Score: 0.875


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Train XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:,1]

auc_xgb = roc_auc_score(y_test, y_prob_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

print("XGBoost AUC:", auc_xgb)
print("XGBoost F1 Score:", f1_xgb)

XGBoost AUC: 0.9815384615384616
XGBoost F1 Score: 0.8979591836734694


Comparison using AUC and F1-score

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost"],
    "AUC": [auc_log, auc_xgb],
    "F1 Score": [f1_log, f1_xgb]
})

print(comparison)

                 Model       AUC  F1 Score
0  Logistic Regression  0.968205  0.875000
1              XGBoost  0.981538  0.897959


**(b) Remove Experience_Years and Retrain Models**

In [ ]:
X_no_exp = X.drop(columns=['Experience_Years'])

from sklearn.model_selection import train_test_split

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_no_exp,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
log_model2 = LogisticRegression(max_iter=1000)

log_model2.fit(X_train2, y_train2)

y_pred_log2 = log_model2.predict(X_test2)
y_prob_log2 = log_model2.predict_proba(X_test2)[:,1]

auc_log2 = roc_auc_score(y_test2, y_prob_log2)
f1_log2 = f1_score(y_test2, y_pred_log2)

print("Logistic Regression (No Experience) AUC:", auc_log2)
print("Logistic Regression (No Experience) F1:", f1_log2)

Logistic Regression (No Experience) AUC: 0.9692307692307693
Logistic Regression (No Experience) F1: 0.8979591836734694


In [ ]:
xgb_model2 = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_model2.fit(X_train2, y_train2)

y_pred_xgb2 = xgb_model2.predict(X_test2)
y_prob_xgb2 = xgb_model2.predict_proba(X_test2)[:,1]

auc_xgb2 = roc_auc_score(y_test2, y_prob_xgb2)
f1_xgb2 = f1_score(y_test2, y_pred_xgb2)

print("XGBoost (No Experience) AUC:", auc_xgb2)
print("XGBoost (No Experience) F1:", f1_xgb2)

XGBoost (No Experience) AUC: 0.9815384615384615
XGBoost (No Experience) F1: 0.8979591836734694
